# From prompt to executable specification

Classify mixed-authority prose, repair weak requirements, route decisions, and validate traceability before an implementation agent writes code. This notebook is offline, deterministic, and uses only the Python standard library.

## Learning contract

You will make a judgment before seeing each reference result. The scenario is fictional. Do not treat its policies, latency target, or architecture decision as advice for a real underwriting system.

In [ ]:
from pathlib import Path
import runpy

lesson_dir = Path.cwd()
if not (lesson_dir / 'lab.py').exists():
    lesson_dir = Path('curriculum/beginner/02-from-prompt-to-executable-specification')
lab = runpy.run_path(str(lesson_dir / 'lab.py'), run_name='course02_lab')
print('Loaded Course 02 lab from', lesson_dir)

## Scenario: AI-1842

A product ticket asks for two-policy comparison, the existing RAG pipeline, accurate explanations, caching, possible human approval, latency under three seconds, and Redis. Before running the next cell, write down which parts are intent, requirements, ambiguity, policy leads, and design suggestions.

In [ ]:
statements = lab['ticket_statements']()
for item in statements:
    print(f'{item.id}: {item.text}')

## Experiment 1 — keyword baseline

Predict which mistakes a classifier will make if it assumes that `use`, `Redis`, `RAG`, or `cache` means requirement. Then run it.

In [ ]:
for item in statements:
    result = lab['keyword_classify'](item)
    print(item.id, '→', result.artifact_type.value, result.reason_codes)

The baseline promotes implementation language into obligations and misses hearsay and vague qualities. Its output is repeatable but not trustworthy because it has no model of authority.

## Experiment 2 — authority-aware routing

For each row, predict whether it should be kept, clarified, verified, moved, or rejected.

In [ ]:
rows = lab['classification_summary'](statements)
for row in rows:
    print(f"{row['id']}: {row['artifact_type']:<16} clarify={row['needs_clarification']} {row['reason_codes']}")

The result is a routing recommendation, not an automatic approval. In particular, Sarah's comment is a lead to an authority source; it is not itself policy.

## Experiment 3 — same words, different authority

Predict the destination of `Use Bedrock` when it comes from an informal ticket versus an approved platform policy.

In [ ]:
RawStatement = lab['RawStatement']; Authority = lab['Authority']
classify = lab['classify_statement']
informal = RawStatement('X1', 'Use Bedrock.', 'feature ticket', 'product manager', Authority.INFORMAL)
policy = RawStatement('X2', 'Use Bedrock.', 'AI platform standard rev 7', 'AI platform owner', Authority.POLICY)
for item in (informal, policy):
    result = classify(item)
    print(item.id, '→', result.artifact_type.value, result.lifecycle.value, result.reason_codes)

The first is a design suggestion requiring clarification; the second is a living constraint. Provenance and decision rights changed the classification even though the words did not.

## Experiment 4 — requirement quality

Predict the findings for vague accuracy, unqualified latency, and an implementation-laden requirement.

In [ ]:
RequirementCandidate = lab['RequirementCandidate']
candidates = [
    RequirementCandidate('R1', 'The response shall be accurate.', 'product owner', 'ticket'),
    RequirementCandidate('R2', 'Latency shall be under 3 seconds.', 'service owner', 'ticket', measurement='under 3 seconds'),
    RequirementCandidate('R3', 'Use Redis so comparisons are fast.', 'product owner', 'ticket'),
]
for candidate in candidates:
    print(candidate.id, [f.code for f in lab['validate_requirement'](candidate)])

Repair R1 with inspectable citation and abstention behavior. Repair R2 with a statistic, measurement boundary, workload, and scope. Move Redis to design or an ADR unless an authoritative constraint owns it.

## Experiment 5 — route decision rights

Predict which decision an agent may make, which needs an ADR proposal, and which must go to a policy owner.

In [ ]:
DecisionCandidate = lab['DecisionCandidate']
decisions = [
    DecisionCandidate('D1', 'Name a local helper', 'local', True, Authority.ENGINEERING),
    DecisionCandidate('D2', 'Create a persistent comparison cache', 'data-retention', False, Authority.ENGINEERING),
    DecisionCandidate('D3', 'Waive human review', 'security', False, Authority.POLICY, True),
]
for item in decisions:
    print(item.id, '→', lab['route_decision'](item))

## Experiment 6 — cross-artifact failure injection

The bad stack puts Redis in a vague requirement, treats hearsay as policy, provides a malformed ADR, links work to unknown intent, lets a test define the requirement, and grants an exception through agent instructions. Predict the stop codes.

In [ ]:
bad = lab['failure_stack']()
bad_findings = lab['validate_stack'](bad)
for finding in bad_findings:
    print(f'{finding.severity.upper():<5} {finding.code:<36} {finding.artifact}: {finding.message}')

## Experiment 7 — reference stack and denominators

Inspect the stack only after writing your own decomposition. A clean structural result means the declared links are coherent; it does not prove the fictional product is production-ready.

In [ ]:
reference = lab['reference_stack']()
print('findings:', lab['validate_stack'](reference))
print('traceability:', lab['traceability_metrics'](reference))

Both metrics show covered and total requirement counts. `100%` without its denominator would hide whether requirements disappeared from scope. Structural coverage also says nothing about whether later evidence passes.

## Your extension

1. Add a requirement with a source but no owner. Explain the stop.
2. Add an unlinked evaluation. Explain why useful evidence can still be ungoverned.
3. Draft an alternative ADR that selects retrieval-result caching; identify the additional controls and evidence it would require.
4. Rewrite one scenario as an invariant and explain what the scenario still contributes.
5. Identify a low-risk change for which this full stack would be disproportionate.

## Exit check

You are ready to continue when you can explain why classification depends on authority; repair a vague obligation; route architecture and policy decisions; separate tasks from outcomes and tests from specifications; and report traceability with explicit denominators.